# Getting Started with imputation-methods

Welcome! This tutorial will help you get started with the imputation-methods library.

## What You'll Learn

1. Understanding missing data
2. Loading and preparing data
3. Applying basic imputation methods
4. Evaluating imputation quality
5. Choosing the right method for your data

## Prerequisites

- Basic Python knowledge
- Familiarity with pandas DataFrames
- Understanding of NumPy arrays

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import imputation methods
from imputation_methods import (
    MeanImputer,
    MedianImputer,
    KNNImputerMethod,
    rmse,
    mae,
)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully!")

## 1. Understanding Missing Data

Missing data is common in real-world datasets. Let's create a simple example to understand it better.

In [ ]:
# Create a simple dataset with missing values
data = pd.DataFrame({
    'temperature': [20.5, 21.3, np.nan, 19.8, 22.1, np.nan, 20.9],
    'humidity': [65, 70, 68, np.nan, 72, 69, np.nan],
    'pressure': [1013, np.nan, 1015, 1014, np.nan, 1016, 1013]
})

print("Original dataset:")
print(data)
print("\nMissing values per column:")
print(data.isna().sum())
print(f"\nTotal missing: {data.isna().sum().sum()} out of {data.size} values")

In [ ]:
# Visualize missing data pattern
plt.figure(figsize=(10, 4))
sns.heatmap(data.isna(), cbar=False, cmap='RdYlGn_r', yticklabels=True)
plt.title('Missing Data Pattern (Red = Missing)', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.ylabel('Samples')
plt.tight_layout()
plt.show()

## 2. Basic Imputation Methods

Let's start with the simplest imputation methods: mean and median.

### Mean Imputation

Replaces missing values with the column mean.

In [ ]:
# Apply mean imputation
mean_imputer = MeanImputer()
data_mean = mean_imputer.impute(data)

print("Data after mean imputation:")
print(data_mean)
print("\nNo more missing values:", data_mean.isna().sum().sum() == 0)

### Median Imputation

Replaces missing values with the column median. More robust to outliers than mean.

In [ ]:
# Apply median imputation
median_imputer = MedianImputer()
data_median = median_imputer.impute(data)

print("Data after median imputation:")
print(data_median)

### K-Nearest Neighbors (KNN) Imputation

Uses similar observations to impute missing values. More sophisticated than mean/median.

In [ ]:
# Apply KNN imputation
knn_imputer = KNNImputerMethod(k=3)
data_knn = knn_imputer.impute(data)

print("Data after KNN imputation:")
print(data_knn)

## 3. Comparing Results

Let's compare how different methods imputed the same missing values.

In [ ]:
# Compare imputed values for temperature column
comparison = pd.DataFrame({
    'Original': data['temperature'],
    'Mean': data_mean['temperature'],
    'Median': data_median['temperature'],
    'KNN': data_knn['temperature']
})

print("Comparison of imputed values (temperature column):")
print(comparison)
print("\nNote: Rows 2 and 5 had missing values")

## 4. Real-World Example: House Prices

Let's work with a more realistic dataset.

In [ ]:
# Create a realistic house price dataset
np.random.seed(42)
n_samples = 100

# Generate complete data
house_data_complete = pd.DataFrame({
    'square_feet': np.random.normal(1500, 500, n_samples),
    'bedrooms': np.random.randint(1, 6, n_samples),
    'age_years': np.random.randint(0, 50, n_samples),
    'price': np.random.normal(300000, 100000, n_samples)
})

# Introduce missing values (15% missing)
house_data = house_data_complete.copy()
mask = np.random.rand(*house_data.shape) < 0.15
house_data[mask] = np.nan

print(f"Dataset shape: {house_data.shape}")
print(f"\nMissing values:")
print(house_data.isna().sum())
print(f"\nTotal missing: {house_data.isna().sum().sum()} ({house_data.isna().sum().sum() / house_data.size * 100:.1f}%)")

In [ ]:
# Visualize the data distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(house_data.columns):
    axes[idx].hist(house_data[col].dropna(), bins=20, alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'{col.replace("_", " ").title()} Distribution', fontweight='bold')
    axes[idx].set_xlabel(col.replace('_', ' ').title())
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Apply different imputation methods
methods = {
    'Mean': MeanImputer(),
    'Median': MedianImputer(),
    'KNN (k=5)': KNNImputerMethod(k=5)
}

imputed_results = {}
for name, imputer in methods.items():
    print(f"Applying {name} imputation...")
    imputed_results[name] = imputer.impute(house_data)
    print(f"  ✓ Complete")

## 5. Evaluating Imputation Quality

Since we have the original complete data, we can evaluate how well each method performed.

In [ ]:
# Calculate error metrics on the hidden cells only. Observed cells are copied
# unchanged, so scoring the whole table would make every method look better.
missing = house_data.isna().to_numpy()
true_values = pd.Series(house_data_complete.to_numpy()[missing])

results = []

for method_name, imputed_data in imputed_results.items():
    imputed_values = pd.Series(imputed_data.to_numpy()[missing])
    results.append({
        'Method': method_name,
        'RMSE': rmse(true_values, imputed_values),
        'MAE': mae(true_values, imputed_values)
    })

results_df = pd.DataFrame(results)
print("\nImputation Quality Metrics (imputed cells only):")
print("="*50)
print(results_df.to_string(index=False))
print("\nLower values indicate better imputation quality.")
print("Errors are in original units, so the price column dominates the scores.")

In [ ]:
# Visualize the comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# RMSE comparison
ax1.bar(results_df['Method'], results_df['RMSE'], color=['#3498db', '#e74c3c', '#2ecc71'])
ax1.set_ylabel('RMSE', fontsize=12)
ax1.set_title('Root Mean Squared Error', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
for i, v in enumerate(results_df['RMSE']):
    ax1.text(i, v, f'{v:.2f}', ha='center', va='bottom')

# MAE comparison
ax2.bar(results_df['Method'], results_df['MAE'], color=['#3498db', '#e74c3c', '#2ecc71'])
ax2.set_ylabel('MAE', fontsize=12)
ax2.set_title('Mean Absolute Error', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
for i, v in enumerate(results_df['MAE']):
    ax2.text(i, v, f'{v:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 6. Key Takeaways

### When to Use Each Method:

1. **Mean Imputation**
   - ✅ Fast and simple
   - ✅ Good for normally distributed data
   - ❌ Reduces variance
   - ❌ Sensitive to outliers

2. **Median Imputation**
   - ✅ Robust to outliers
   - ✅ Good for skewed distributions
   - ❌ Reduces variance
   - ❌ Ignores relationships between features

3. **KNN Imputation**
   - ✅ Preserves relationships between features
   - ✅ More accurate than simple methods
   - ❌ Slower for large datasets
   - ❌ Requires choosing k parameter

### General Guidelines:

- Start with simple methods (mean/median) to establish a baseline
- Use KNN for better accuracy if computational cost is acceptable
- Always validate imputation quality when possible
- Consider the nature of your data (distribution, outliers, relationships)
- Document your imputation strategy for reproducibility

## 7. Next Steps

Now that you understand the basics, explore:

- **02_method_comparison.ipynb** - Comparison of common methods across missing-data patterns
- **[examples/](../examples/)** - Runnable scripts for time series, ML pipelines and custom imputers
- **demo_imputation.ipynb** - Quick demonstration of key methods

### Additional Resources:

- [README.md](../README.md) - Full documentation
- [CONTRIBUTING.md](../CONTRIBUTING.md) - How to contribute
- [API Documentation](../src/imputation_methods/) - Method details

## Practice Exercise

Try this exercise to test your understanding:

1. Create your own dataset with missing values
2. Apply at least 3 different imputation methods
3. Compare the results
4. Explain which method works best for your data and why

In [ ]:
# Your code here
